# 98 Codomain Experiments

This notebook is optional and intentionally outside the main sequence.

Use it to prototype codomain changes before promoting anything into the main pipeline.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidates = [cwd, *list(cwd.parents[:3])]
PROJECT_ROOT = next((path for path in candidates if (path / "working" / "lib" / "pipeline.py").exists()), cwd)
LIB_ROOT = PROJECT_ROOT / "working" / "lib"
if str(LIB_ROOT) not in sys.path:
    sys.path.insert(0, str(LIB_ROOT))

import pipeline as xp

xp.ensure_working_tree()
print("Project root:", PROJECT_ROOT)
display(xp.list_subjects())


In [ ]:
display(xp.codomain_improvement_candidates())


In [ ]:
SUBJECT = "costco"
VARIANT_NAME = "costco_adjacent_interest_v1"
DROP_RETWEETS = False
DROP_REPLIES = False
RUN_COMPARISON_GRID = True
defaults = xp.codomain_filter_defaults(SUBJECT)
INCLUDE_TERMS = defaults.get("include_terms", [])
EXCLUDE_TERMS = defaults.get("exclude_terms", [])
MAX_POSTS_PER_USER = defaults.get("max_posts_per_user", 12)
MIN_INCLUDE_MATCHES = defaults.get("min_include_matches", 1)
MAX_EXCLUDE_MATCHES = defaults.get("max_exclude_matches", 0)
MIN_RELEVANCE_SCORE = defaults.get("min_relevance_score", 1)
MIN_AUTHOR_INCLUDE_HITS = max(2, defaults.get("min_author_include_hits", 2))
MIN_AUTHOR_FOCUS_SHARE = defaults.get("min_author_focus_share", 0.20)
MAX_AUTHOR_URL_SHARE = 0.75
DROP_DIRECT_SUBJECT_POSTS = True
DROP_LINK_HEAVY_NOISE = True
MAX_URLS = 1
DROP_CASHTAGS = True
MIN_ALPHA_WORDS_WITH_URL = 6
DOMAIN_SIMILARITY_QUANTILE = None
DOMAIN_SIMILARITY_MIN = None


In [ ]:
subject = xp.get_subject_config(SUBJECT)
codomain_df = xp.load_stage_table(subject, "01_ingest", "codomain", "corpus", "csv")

if RUN_COMPARISON_GRID:
    comparison_specs = [
        {
            "variant_name": f"{SUBJECT}_qualified_users_all_posts",
            "drop_direct_subject_posts_from_kept_users": False,
            "drop_link_heavy_noise": False,
            "max_author_url_share": None,
            "domain_similarity_quantile": None,
            "domain_similarity_min": None,
        },
        {
            "variant_name": f"{SUBJECT}_adjacent_interest_posts",
            "drop_direct_subject_posts_from_kept_users": True,
            "drop_link_heavy_noise": False,
            "max_author_url_share": None,
            "domain_similarity_quantile": None,
            "domain_similarity_min": None,
        },
        {
            "variant_name": f"{SUBJECT}_adjacent_interest_clean",
            "drop_direct_subject_posts_from_kept_users": True,
            "drop_link_heavy_noise": True,
            "max_author_url_share": MAX_AUTHOR_URL_SHARE,
            "domain_similarity_quantile": None,
            "domain_similarity_min": None,
        },
    ]
    comparison_runs = []
    comparison_rows = []
    for spec in comparison_specs:
        result = xp.build_codomain_variant(
            subject,
            codomain_df,
            variant_name=spec["variant_name"],
            include_terms=INCLUDE_TERMS,
            exclude_terms=EXCLUDE_TERMS,
            drop_retweets=DROP_RETWEETS,
            drop_replies=DROP_REPLIES,
            max_posts_per_user=MAX_POSTS_PER_USER,
            min_include_matches=MIN_INCLUDE_MATCHES,
            max_exclude_matches=MAX_EXCLUDE_MATCHES,
            min_relevance_score=MIN_RELEVANCE_SCORE,
            min_author_include_hits=MIN_AUTHOR_INCLUDE_HITS,
            min_author_focus_share=MIN_AUTHOR_FOCUS_SHARE,
            max_author_url_share=spec["max_author_url_share"],
            drop_direct_subject_posts_from_kept_users=spec["drop_direct_subject_posts_from_kept_users"],
            drop_link_heavy_noise=spec["drop_link_heavy_noise"],
            max_urls=MAX_URLS,
            drop_cashtags=DROP_CASHTAGS,
            min_alpha_words_with_url=MIN_ALPHA_WORDS_WITH_URL,
            domain_similarity_quantile=spec["domain_similarity_quantile"],
            domain_similarity_min=spec["domain_similarity_min"],
        )
        comparison_runs.append(result)
        final_stage = result["audit"].iloc[-1].to_dict()
        comparison_rows.append(
            {
                "variant": result["variant"],
                "rows": final_stage.get("rows"),
                "unique_authors": final_stage.get("unique_authors"),
                "drop_direct_subject_posts": spec["drop_direct_subject_posts_from_kept_users"],
                "drop_link_heavy_noise": spec["drop_link_heavy_noise"],
                "max_author_url_share": spec["max_author_url_share"],
                "mean_domain_similarity": final_stage.get("mean_domain_similarity"),
                "threshold": result.get("similarity_summary", {}).get("threshold"),
                "manifest_path": result["manifest_path"],
            }
        )
    display(pd.DataFrame(comparison_rows))
else:
    comparison_runs = []
    print("Comparison grid skipped.")


In [ ]:
experiment = xp.build_codomain_variant(
    subject,
    codomain_df,
    variant_name=VARIANT_NAME,
    include_terms=INCLUDE_TERMS,
    exclude_terms=EXCLUDE_TERMS,
    drop_retweets=DROP_RETWEETS,
    drop_replies=DROP_REPLIES,
    max_posts_per_user=MAX_POSTS_PER_USER,
    min_include_matches=MIN_INCLUDE_MATCHES,
    max_exclude_matches=MAX_EXCLUDE_MATCHES,
    min_relevance_score=MIN_RELEVANCE_SCORE,
    min_author_include_hits=MIN_AUTHOR_INCLUDE_HITS,
    min_author_focus_share=MIN_AUTHOR_FOCUS_SHARE,
    max_author_url_share=MAX_AUTHOR_URL_SHARE,
    drop_direct_subject_posts_from_kept_users=DROP_DIRECT_SUBJECT_POSTS,
    drop_link_heavy_noise=DROP_LINK_HEAVY_NOISE,
    max_urls=MAX_URLS,
    drop_cashtags=DROP_CASHTAGS,
    min_alpha_words_with_url=MIN_ALPHA_WORDS_WITH_URL,
    domain_similarity_quantile=DOMAIN_SIMILARITY_QUANTILE,
    domain_similarity_min=DOMAIN_SIMILARITY_MIN,
)
experiment


In [ ]:
display(experiment["audit"])
print("Filters")
display(pd.DataFrame([experiment.get("filters", {})]))
print("Similarity summary")
display(pd.DataFrame([experiment.get("similarity_summary", {})]))
print("Noise filter meta")
display(pd.DataFrame([experiment.get("noise_filter_meta", {})]))
display(experiment["author_summary"].head(20))


In [ ]:
print("Kept sample")
display(experiment["kept_sample"])
print("Dropped sample")
display(experiment["dropped_sample"])


In [ ]:
preview_subject = subject if "subject" in globals() else xp.get_subject_config(SUBJECT)
_ = xp.show_stage_figures(preview_subject, "98_experiments")
